# Testing-Effect Question Generation (QG) — Data Prep

No existing implementation to convert here — `src/models/` and `src/train/`
are still just `.gitkeep` placeholders, and the Obsidian project Hub only
had this as a TODO ("testing effect (QG, C) · confirm remaining pipeline
stages' team-member assignment"). This notebook builds the QG training data
from scratch, mirroring `04_rehearsal_maintenance_prep.ipynb`'s structure.

- **input** = `answer: {answer} context: {context}` — the reverse of
  normal QA (given context + question, produce an answer): here the model
  is given the context and a specific answer span, and must produce a
  question whose answer is that span. This is what lets the testing-effect
  stage generate quiz questions targeting a specific fact in a chunk.
- **target** = the reference question.

Uses `squad` (v1.1) — SQuAD ships (context, question, answer) triples
directly, so no oracle-extraction step is needed the way
`cnn_dailymail`'s abstractive-only summaries required for the A/B rehearsal
data (notebook `04`). SQuAD contexts are Wikipedia paragraphs, already close to
chunk scale (measured: 50th/90th/99th percentile word counts are
143/239/312 — comfortably under `configs/chunking.yaml`'s `max_words=350`),
so no truncation step is needed either.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `squad`

`squad` (v1.1) has official `train`/`validation` splits with `context`,
`question`, `answers` (`{"text": [...], "answer_start": [...]}`) fields.
Each training row has exactly one answer; we just take `answers.text[0]`.

Unlike `cnn_dailymail` (notebook `04`), SQuAD has no public `test` split — its actual
test set answers are held out for the official leaderboard, not
distributed. So `validation` is split in half here: the first half is used
for training-time model selection (as `val`), the second half is held out
and only touched once, at the end of `09_qg_testing_effect_train.ipynb`, for
final reporting (as `test`) — same role separation as `04`'s official
3-way split, just carved out manually since SQuAD doesn't provide it.

In [2]:
MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300

train_raw = load_dataset("squad", split=f"train[:{MAX_TRAIN_EXAMPLES}]")
val_test_raw = load_dataset("squad", split=f"validation[:{MAX_VAL_EXAMPLES + MAX_TEST_EXAMPLES}]")
val_raw = val_test_raw.select(range(MAX_VAL_EXAMPLES))
test_raw = val_test_raw.select(range(MAX_VAL_EXAMPLES, len(val_test_raw)))
print(f"train: {len(train_raw)}, validation: {len(val_raw)}, test: {len(test_raw)}")

sample = train_raw[0]
print("\nsample keys:", list(sample.keys()))
print("context (first 200 chars):", sample["context"][:200])
print("question:", sample["question"])
print("answers:", sample["answers"])


train: 3000, validation: 300, test: 300

sample keys: ['id', 'title', 'context', 'question', 'answers']
context (first 200 chars): Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


## 2. Build (input, target) pairs — reversed QA

`input_text = "answer: {answer} context: {context}"`, `target_text =
question`. Rows with no answer text (shouldn't happen in SQuAD v1.1's
train/validation splits, but checked defensively) are skipped.

In [3]:
def build_pair(example: dict) -> dict | None:
    answer_texts = example["answers"]["text"]
    if not answer_texts:
        return None
    return {
        "id": example["id"],
        "input_text": f"answer: {answer_texts[0]} context: {example['context']}",
        "target_text": example["question"],
    }


train_pairs = [p for p in (build_pair(ex) for ex in train_raw) if p is not None]
val_pairs = [p for p in (build_pair(ex) for ex in val_raw) if p is not None]
test_pairs = [p for p in (build_pair(ex) for ex in test_raw) if p is not None]
print(f"train pairs: {len(train_pairs)} / {len(train_raw)}")
print(f"val pairs: {len(val_pairs)} / {len(val_raw)}")
print(f"test pairs: {len(test_pairs)} / {len(test_raw)}")

pd.DataFrame(train_pairs)[["input_text", "target_text"]].head(3)


train pairs: 3000 / 3000
val pairs: 300 / 300
test pairs: 300 / 300


,input_text,target_text
0,answer: Saint Bernadette Soubirous context: Ar...,To whom did the Virgin Mary allegedly appear i...
1,answer: a copper statue of Christ context: Arc...,What is in front of the Notre Dame Main Building?
2,answer: the Main Building context: Architectur...,The Basilica of the Sacred heart at Notre Dame...


## 3. Tokenize

`INPUT_MAX_LENGTH`/`TARGET_MAX_LENGTH` are set from measured token-length
percentiles with the `t5-small` tokenizer: input (`answer: ... context:
...`) runs ~213/359/513 tokens at the 50th/90th/99th percentile, and
questions run ~16/22/26. 512 covers the bulk of inputs (a handful of the
longest contexts get truncated at the tail — acceptable for a pilot); 32
comfortably covers question length.

In [4]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 32

tokenizer = AutoTokenizer.from_pretrained("t5-small")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer(
        [p["input_text"] for p in pairs],
        max_length=INPUT_MAX_LENGTH,
        truncation=True,
    )
    targets = tokenizer(
        [p["target_text"] for p in pairs],
        max_length=TARGET_MAX_LENGTH,
        truncation=True,
    )
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
test_dataset = tokenize_pairs(test_pairs)
print(train_dataset)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


## 4. Save

`notebooks/09_qg_testing_effect_train.ipynb` loads these directly —
train+val during training, test only once at the end for final reporting.

In [5]:
OUT_DIR = Path("data/processed/qg_testing_effect")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(test_pairs).to_csv(OUT_DIR / "test_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
print(f"  train_dataset: {len(train_dataset)} rows")
print(f"  val_dataset: {len(val_dataset)} rows")
print(f"  test_dataset: {len(test_dataset)} rows")


Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards): 100%|██████████| 3000/3000 [00:01<00:00, 1684.61 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:01<00:00, 1684.61 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:01<00:00, 1683.21 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

...

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 116422.21 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 132104.06 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 118260.45 examples/s]

Saved to: data/processed/qg_testing_effect
  train_dataset: 3000 rows
  val_dataset: 300 rows
  test_dataset: 300 rows


## Summary

- Source: `squad` (v1.1). `train` sliced to `MAX_TRAIN_EXAMPLES` (3,000);
  `validation` sliced to `MAX_VAL_EXAMPLES + MAX_TEST_EXAMPLES` (600) and
  split in half into `val` (model selection during training) and `test`
  (touched once, at the end, for final reporting) — SQuAD has no public
  test split, so this is carved out manually rather than using an official
  one (contrast with `04`, where `cnn_dailymail`'s official `test` split is
  used directly).
- No oracle-extraction step needed (unlike `04`'s rehearsal-maintenance
  data) — SQuAD gives (context, question, answer) directly, we just reverse
  the usual QA direction.
- No context truncation needed either — SQuAD's Wikipedia-paragraph
  contexts are already within `configs/chunking.yaml`'s chunk-scale budget.
- Next: `09_qg_testing_effect_train.ipynb` trains `t5-small` on this data,
  the same pattern as `05_rehearsal_maintenance_train.ipynb`.